In [3]:
# ⚙️ Global Config & Services (using centralized modules)

import sys
from pathlib import Path
from dotenv import load_dotenv

# Add parent directory to path and change to project root
import os

# Get the notebook's current directory and find project root
notebook_dir = Path.cwd()
if notebook_dir.name == "notebooks":
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

# Change to project root and add to path
os.chdir(project_root)
sys.path.insert(0, str(project_root))

print(f"Working directory: {os.getcwd()}")

from src.services.llm_services import (
    load_config,
    get_llm,
    validate_api_keys,
    print_config_summary,
    get_text_embeddings
)

# Load environment variables
load_dotenv()

# Load configuration from config.yaml (now we're in project root)
config = load_config("src/config/config.yaml")

# Validate API keys
validate_api_keys(config, verbose=True)

# Print summary
print_config_summary(config)


Working directory: c:\Users\viraj\Zuu\Ledger_mind
✅ Config loaded:
  LLM: groq / llama-3.1-8b-instant
  Embeddings: sbert / sentence-transformers/all-MiniLM-L6-v2
  Temperature: 0.2
  Artifacts: ./artifacts


c:\Users\viraj\Zuu\Ledger_mind\src\services\llm_services.py:316: UserWarning: ⚠️  OPENAI_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")
c:\Users\viraj\Zuu\Ledger_mind\src\services\llm_services.py:316: UserWarning: ⚠️  OPENROUTER_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")
c:\Users\viraj\Zuu\Ledger_mind\src\services\llm_services.py:316: UserWarning: ⚠️  GOOGLE_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")
c:\Users\viraj\Zuu\Ledger_mind\src\services\llm_services.py:316: UserWarning: ⚠️  COHERE_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")


In [4]:
# Initialize LLM using factory from llm_services
llm = get_llm(config)
print(f"LLM initialized: {config['llm_provider']} / {config['llm_model']}")

# Verify API key with test completion
print("\n🔍 Testing API connection...")
try:
    test_response = llm.invoke("Say 'API working!' if you can read this.")
    test_msg = test_response.content if hasattr(test_response, 'content') else str(test_response)
    print(f"API key verified: {test_msg[:50]}")
except Exception as e:
    print(f" API key test failed: {e}")
    print("Please check your .env file and API key configuration.")


LLM initialized: groq / llama-3.1-8b-instant

🔍 Testing API connection...
API key verified: API working!


In [5]:
# Load documents using the utility module
from src.utils.document_loader import load_pdf_documents

# Load all PDFs from data directory
pdf_dir = Path(config["data_root"])
documents = load_pdf_documents(pdf_dir)

print(f"\n✅ Loaded {len(documents)} documents total")

✓ Loaded annual_report_2024.pdf: 640,073 chars from 142 pages

✅ Loaded 1 documents total


### Inspect Document Data

In [6]:
# Inspect loaded documents
import re

def inspect_document(doc, sample_lines=10):
    """Display comprehensive document statistics and samples."""
    content = doc['content']
    lines = content.split('\n')
    
    print(f"\n{'='*70}")
    print(f"📄 Document: {doc['source']}")
    print(f"{'='*70}")
    
    # Basic stats
    print(f"\n📊 Statistics:")
    print(f"  Total characters: {len(content):,}")
    print(f"  Total lines: {len(lines):,}")
    print(f"  Total words: {len(content.split()):,}")
    print(f"  Avg line length: {len(content) / max(len(lines), 1):.1f} chars")
    
    # Non-empty lines
    non_empty_lines = [line for line in lines if line.strip()]
    print(f"  Non-empty lines: {len(non_empty_lines):,}")
    
    # Sample from beginning
    print(f"\n📖 First {sample_lines} lines:")
    print("-" * 70)
    for i, line in enumerate(lines[:sample_lines], 1):
        display_line = line[:100] + "..." if len(line) > 100 else line
        print(f"  {i:3d}: {display_line}")
    
    # Sample from middle
    mid_point = len(lines) // 2
    print(f"\n📖 Sample from middle (around line {mid_point}):")
    print("-" * 70)
    for i, line in enumerate(lines[mid_point:mid_point+sample_lines], mid_point+1):
        display_line = line[:100] + "..." if len(line) > 100 else line
        print(f"  {i:3d}: {display_line}")
    
    # Sample from end
    print(f"\n📖 Last {sample_lines} lines:")
    print("-" * 70)
    start_idx = max(0, len(lines) - sample_lines)
    for i, line in enumerate(lines[-sample_lines:], start_idx+1):
        display_line = line[:100] + "..." if len(line) > 100 else line
        print(f"  {i:3d}: {display_line}")
    
    # Detect potential headers/footers
    print(f"\n🔍 Potential Headers/Footers:")
    print("-" * 70)
    
    # Look for page numbers
    page_num_pattern = r'^\s*(?:page\s*)?(\d+)(?:\s+of\s+\d+)?\s*$'
    page_nums = [line.strip() for line in lines if re.match(page_num_pattern, line.strip(), re.IGNORECASE)]
    if page_nums:
        print(f"  Page numbers found: {len(page_nums)} occurrences")
        print(f"    Examples: {', '.join(page_nums[:5])}")
    
    # Look for dates
    date_pattern = r'\d{1,2}[-/]\d{1,2}[-/]\d{2,4}|(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\w*\s+\d{4}'
    dates = [line.strip() for line in lines if re.match(f'^{date_pattern}$', line.strip(), re.IGNORECASE)]
    if dates:
        print(f"  Date footers found: {len(dates)} occurrences")
        print(f"    Examples: {', '.join(dates[:5])}")
    
    # Look for very short lines (potential artifacts)
    very_short = [line.strip() for line in lines if 0 < len(line.strip()) < 3]
    if very_short:
        print(f"  Very short lines (1-2 chars): {len(very_short)} occurrences")
        print(f"    Examples: {', '.join(very_short[:10])}")
    
    # Look for URLs
    url_pattern = r'https?://|www\.'
    urls = [line.strip() for line in lines if re.search(url_pattern, line.strip(), re.IGNORECASE)]
    if urls:
        print(f"  URLs found: {len(urls)} occurrences")
        print(f"    Examples: {urls[0][:60] if urls else ''}")
    
    print()

# Inspect all documents
for doc in documents:
    inspect_document(doc, sample_lines=10)


📄 Document: annual_report_2024.pdf

📊 Statistics:
  Total characters: 640,073
  Total lines: 7,255
  Total words: 98,684
  Avg line length: 88.2 chars
  Non-empty lines: 6,846

📖 First 10 lines:
----------------------------------------------------------------------
    1: On Our Way
    2: 2024 ANNUAL REPORT
    3: Uber’s Mission
    4: We reimagine the way the world moves for the better
    5: We are Uber. The go-getters. The kind of people who are relentless about our  
    6: mission to help people go anywhere and get anything and earn their way. 
    7: Movement is what we power. It’s our lifeblood. It runs through our veins. It’s 
    8: what gets us out of bed each morning. It pushes us to constantly reimagine 
    9: how we can move better. For you. For all the places you want to go. For all the 
   10: things you want to get. For all the ways you want to earn. Across the entire 

📖 Sample from middle (around line 3627):
---------------------------------------------------------

### Clean Data

In [7]:
# Clean documents using the utility module
from src.utils.document_loader import clean_documents, save_cleaned_text

# Clean documents (remove headers/footers)
cleaned_docs = clean_documents(documents, remove_footers=True)

# Save cleaned text files
output_dir = Path(config.get("data_root", "data")) / "cleaned"
save_cleaned_text(cleaned_docs, output_dir)

print(f"\n✅ Cleaned and saved {len(cleaned_docs)} documents")

✓ Cleaned annual_report_2024.pdf: 640,073 → 624,095 chars (2.5% removed)
✓ Saved to data\cleaned\annual_report_2024_cleaned.txt

✅ Cleaned and saved 1 documents


### Fixed Chunking

In [8]:
from  src.utils.text_splitter import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
)

def get_splitter(strategy: str, chunk_size: int = 1500, chunk_overlap: int = 150):
    """
    Return a text splitter based on the specified strategy.
    
    Args:
        strategy: One of "recursive", "fixed", or "sentence"
        chunk_size: Maximum characters per chunk
        chunk_overlap: Overlap between consecutive chunks
        
    Returns:
        A LangChain text splitter instance
    """
    if strategy == "recursive": 
        return RecursiveCharacterTextSplitter(
                                            chunk_size=chunk_size,
                                            chunk_overlap=chunk_overlap,
                                            separators=["\n\n", "\n", ". ", ", ", " ", ""]
                                            )

    elif strategy == "fixed":
        return CharacterTextSplitter(
                                    chunk_size=chunk_size,
                                    chunk_overlap=chunk_overlap,
                                    separator=" "
                                    )

    elif strategy == "sentence":
        pass 

    else:
        raise ValueError("Unknown stratergy")
    



In [9]:
from pathlib import Path
from langchain_core.documents import Document

txt_path = Path("C:/Users/viraj/Zuu/Ledger_mind/data/cleaned/annual_report_2024_cleaned.txt")  # update this path
text = txt_path.read_text(encoding="utf-8")

document = [Document(page_content=text, metadata={"source": str(txt_path)})]
split_results = {}
strategy = "fixed"
splitter = get_splitter(strategy)
chunks = splitter.split_documents(document)

# Add strategy to metadata
for chunk in chunks:
    chunk.metadata["splitter"] = strategy

split_results[strategy] = chunks
print(f"✅ {strategy:10s}: {len(chunks):4d} chunks")

print(f"\nExample chunk ({strategy}):")
print(f"  Length: {len(split_results[strategy][0].page_content)} chars")
print(f"  Content: {split_results[strategy][0].page_content[:150]}...")
print(f"  Metadata: {split_results[strategy][0].metadata}")

✅ fixed     :  464 chunks

Example chunk (fixed):
  Length: 1490 chars
  Content: On Our Way
2024 ANNUAL REPORT
Uber’s Mission
We reimagine the way the world moves for the better
We are Uber. The go-getters. The kind of people who a...
  Metadata: {'source': 'C:\\Users\\viraj\\Zuu\\Ledger_mind\\data\\cleaned\\annual_report_2024_cleaned.txt', 'chunk_index': 0, 'splitter': 'fixed'}


### Chroma DB

In [10]:
from langchain_chroma import Chroma

embeddings = get_text_embeddings(config)

from src.services.llm_services import load_config
config = load_config("src/config/config.yaml")

chroma_root = Path(config["artifacts_root"]) / "chroma"
chroma_root.mkdir(parents=True, exist_ok=True)

vectorstores = {}
strategy = "fixed"


collection_name = f"langchain_{strategy}"
persist_dir = str(chroma_root / collection_name)
    
print(f"Building collection: {collection_name}...")
    
vectorstore = Chroma.from_documents(
                                    documents=split_results[strategy],
                                    embedding=embeddings,
                                    collection_name=collection_name,
                                    persist_directory=persist_dir
                                    )
    
print(f"  ✅ Persisted to {persist_dir}")
print(f"  ✅ {len(split_results[strategy])} chunks embedded")

vectorstores[strategy] = vectorstore

print(f"\n✅ All collection built!")


ModuleNotFoundError: No module named 'langchain_huggingface'

In [ ]:


manifests_dir = Path(config["artifacts_root"]) / "manifests"
manifests_dir.mkdir(parents=True, exist_ok=True)

for strategy in strategies:
    manifest = {
        "collection_name": f"langchain_{strategy}",
        "framework": "langchain",
        "strategy": strategy,
        "embedding_model": config["text_emb_model"],
        "embedding_provider": config["text_emb_provider"],
        "normalize": config["normalize_embeddings"],
        "chunk_size": 800,
        "chunk_overlap": 150,
        "num_chunks": len(split_results[strategy]),
        "num_documents": len(documents),
        "created_at": datetime.now().isoformat(),
    }
    
    manifest_path = manifests_dir / f"langchain_{strategy}.json"
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)
    
    print(f"✅ Manifest saved: {manifest_path.name}")


### QA Session